# Moon AI — Teacher Model Tesla T4 Compatibility & Benchmark Notebook
**Chief ML Systems Architect:** Prem Gaikwad (India 🇮🇳)  
**Ownership Policy:** `TEACHER_STUDENT_RANDOM_INIT` (Policy B)  
**Stage:** 2.1.5 — Local Teacher Installation, Qualification & First Verified Corpus Generation  

This notebook verifies the hardware sizing, VRAM allocation, and throughput of the qualified offline teacher model (`Qwen2.5-7B-Instruct-GGUF` Q4_K_M) on a Google Colab NVIDIA Tesla T4 GPU without any API keys or credentials.

## 1. Verify Compute Hardware (Tesla T4 GPU)

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Total VRAM:     {props.total_memory / (1024**3):.2f} GB ({props.total_memory / (1024**2):.1f} MB)")

## 2. Check Storage & Download Model Without Credentials

In [ ]:
# Zero credentials: No Hugging Face tokens, no API keys
!python scripts/download_teacher_model.py --check-only
!python scripts/download_teacher_model.py

## 3. Run T4 Compatibility Benchmark & VRAM Scaling

In [ ]:
!python scripts/benchmark_teacher_t4.py --context-len 4096 --repetitions 5

## 4. Verify Policy B: Random Student Initialization
Confirm that the student Moon GPT initializes strictly from random Gaussian scratch weights and receives zero teacher parameters.

In [ ]:
import os
import torch
from training.chat.train_chat import initialize_scratch_student

model = initialize_scratch_student()
total_params = sum(p.numel() for p in model.parameters())
print(f"[OK] Student initialized from random Gaussian scratch: {total_params:,} parameters")
print("[OK] Teacher/Student Isolation: Verified zero pretrained checkpoint weights.")